# Fine-tuning SentiFlow v11 (config v6 + A100)

## Config gagnante
- xlm-roberta-large, freeze 20/24
- 6 classes (PAS de neutre, gere par seuil confiance cote app)
- dair-ai/emotion uniquement
- Oversampling 70%, class weights
- LR=1e-5, cosine, bf16, batch=32
- 15 epochs, early stopping

In [ ]:
!pip install -q transformers datasets accelerate scikit-learn seaborn

In [ ]:
import torch
print(f'GPU: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

In [ ]:
from datasets import load_dataset, concatenate_datasets
import collections
import numpy as np

dataset = load_dataset('dair-ai/emotion')
label_names = dataset['train'].features['label'].names
counts = collections.Counter(dataset['train']['label'])

print(f'Dataset: {len(dataset["train"])} train, {len(dataset["validation"])} val, {len(dataset["test"])} test')
for idx, name in enumerate(label_names):
    print(f'  {name}: {counts[idx]} ({counts[idx]/len(dataset["train"])*100:.1f}%)')

In [ ]:
# Oversampling 70%
max_count = max(counts.values())
target_count = int(max_count * 0.7)

train_data = dataset['train']
oversampled_parts = [train_data]

for label_idx in range(len(label_names)):
    current = counts[label_idx]
    if current < target_count:
        needed = target_count - current
        class_data = train_data.filter(lambda x: x['label'] == label_idx)
        n_repeats = needed // len(class_data) + 1
        repeated = concatenate_datasets([class_data] * n_repeats).select(range(needed))
        oversampled_parts.append(repeated)
        print(f'  {label_names[label_idx]}: {current} -> {current + needed}')
    else:
        print(f'  {label_names[label_idx]}: {current} (ok)')

train_final = concatenate_datasets(oversampled_parts).shuffle(seed=42)
print(f'\nApres oversampling: {len(train_final)} exemples')

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = 'xlm-roberta-large'
num_labels = 6
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

FREEZE_LAYERS = 20
for name, param in model.named_parameters():
    if 'roberta.encoder.layer' in name:
        layer_num = int(name.split('layer.')[1].split('.')[0])
        if layer_num < FREEZE_LAYERS:
            param.requires_grad = False
    elif 'roberta.embeddings' in name:
        param.requires_grad = False

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'{model_name}: freeze {FREEZE_LAYERS}/24, {trainable/1e6:.0f}M entrainables ({trainable/total*100:.0f}%)')

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train = train_final.map(tokenize_function, batched=True)
tokenized_val = dataset['validation'].map(tokenize_function, batched=True)
tokenized_test = dataset['test'].map(tokenize_function, batched=True)
print(f'Tokenise: train={len(tokenized_train)}, val={len(tokenized_val)}, test={len(tokenized_test)}')

In [ ]:
import torch
import torch.nn as nn
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

labels_array = np.array(dataset['train']['label'])
class_weights = compute_class_weight('balanced', classes=np.unique(labels_array), y=labels_array)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print('Class weights:')
for i, name in enumerate(label_names):
    print(f'  {name}: {class_weights[i]:.2f}')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        weights = class_weights_tensor.to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_weighted': f1_score(labels, preds, average='weighted'),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

run_name = 'v11_final'

training_args = TrainingArguments(
    output_dir=f'./emotion_model_{run_name}',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=1e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.06,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=15,
    weight_decay=0.01,
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model='f1_weighted',
    greater_is_better=True,
    logging_steps=50,
    bf16=True,
    seed=42,
    dataloader_num_workers=4,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print(f'Trainer [{run_name}] pret')

In [ ]:
print(f'Entrainement [{run_name}]...')
trainer.train()
print('Termine!')

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

LABELS_FR = ['tristesse', 'joie', 'amour', 'colere', 'peur', 'surprise']

predictions = trainer.predict(tokenized_test)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = np.array(dataset['test']['label'])

print('=' * 60)
print(f'RESULTATS [{run_name}]')
print('=' * 60)
print(f'Accuracy:    {accuracy_score(y_true, y_pred):.1%}')
print(f'F1 weighted: {f1_score(y_true, y_pred, average="weighted"):.1%}')
print(f'F1 macro:    {f1_score(y_true, y_pred, average="macro"):.1%}')
print()
print(classification_report(y_true, y_pred, target_names=LABELS_FR, digits=3))

print('\nHISTORIQUE:')
print('v1 (base, T4):     Acc=91.6%, F1m=87.5%')
print('v6 (large, T4):    Acc=91.7%, F1m=88.3%')
print('v11 (large, A100):  voir ci-dessus')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS_FR, yticklabels=LABELS_FR)
plt.xlabel('Prediction')
plt.ylabel('Vrai label')
plt.title(f'Matrice de confusion [{run_name}]')
plt.tight_layout()
plt.savefig(f'confusion_matrix_{run_name}.png', dpi=150)
plt.show()

In [ ]:
trainer.save_model('./sentiflow_emotion_model')
tokenizer.save_pretrained('./sentiflow_emotion_model')
print('Modele sauvegarde')

In [ ]:
from transformers import pipeline

classifier = pipeline('text-classification', model='./sentiflow_emotion_model', top_k=None, device=0)
LABELS = ['tristesse', 'joie', 'amour', 'colere', 'peur', 'surprise']

tests = [
    'Je suis tellement heureux!',
    'Je suis triste et deprime',
    'C est inadmissible, je suis furieux!',
    'Je t aime de tout mon coeur',
    'J ai peur de ce qui va arriver',
    'Wow je m attendais pas a ca!',
    'jujutsu kaisen',
    'BTS concert',
    'The weather is nice today',
    'I am so happy and excited!',
    'I am so angry right now',
    'This is scary and frightening',
    'just had lunch',
    'meeting at 3pm',
]

print('\nTESTS MANUELS')
print('=' * 60)
for text in tests:
    result = classifier(text)[0]
    top = max(result, key=lambda x: x['score'])
    idx = int(top['label'].split('_')[-1])
    top3 = sorted(result, key=lambda x: -x['score'])[:3]
    s = ' | '.join(f"{LABELS[int(r['label'].split('_')[-1])]}={r['score']:.0%}" for r in top3)
    print(f'\n"{text}"')
    print(f'  -> {LABELS[idx].upper()} ({top["score"]:.0%}) [{s}]')

In [ ]:
# Push sur HuggingFace
from huggingface_hub import login

login()

model.push_to_hub('davidiabd2/SENTIFLOW_TWEET_FEELING')
tokenizer.push_to_hub('davidiabd2/SENTIFLOW_TWEET_FEELING')
print('Modele uploade sur https://huggingface.co/davidiabd2/SENTIFLOW_TWEET_FEELING')

In [ ]:
!zip -r sentiflow_emotion_model.zip sentiflow_emotion_model/
from google.colab import files
files.download('sentiflow_emotion_model.zip')